# Used Car Price Prediction

## 01. Setup



In [1]:
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# File paths
TRAIN_PATH = "/content/dataset-2.csv"
TARGET_COLUMN = "price"                # <-- target variable: selling price
EVALUATION_SPLIT_PATH = "./evaluation_split_task2.csv"  # organizers only
RANDOM_STATE = 42

## 1. Load the Data

Load the dataset from `TRAIN_PATH` into a dataframe.

In [2]:
df = pd.read_csv(TRAIN_PATH)

# Target column is stored as text like "$27,890" -> convert to a number (formatting only, no rows removed)
if not pd.api.types.is_numeric_dtype(df[TARGET_COLUMN]):
    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN].astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce"
    )

print("Shape:", df.shape)
df.head()

Shape: (3207, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,27890
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,50000
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,33995
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,47645
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,29990


## 2. Explore and Clean the Dataset


In [3]:
def data_cleaning(df):
    """
    Cleaning rules (follows competition rules):
      - NEVER drops rows and NEVER touches the price target.
      - Fixes types / inconsistent labels only. Missing values are filled later inside the model Pipeline
        (fitted on train only -> no leakage).
    Documented decisions:
      * duplicates + extreme price outliers are removed ONLY from the train split (remove_bad_training_rows, Section 5).
      * 'milage' ("81,000 mi.") -> numeric 'mileage'.
      * fuel_type placeholders ('–', 'not supported') -> 'Unknown'.
      * accident -> 1/0, clean_title -> 1/0 (missing stays NaN / 0).
    """
    df = df.copy()

    # column name typo in the raw file
    if "milage" in df.columns:
        df = df.rename(columns={"milage": "mileage"})

    # mileage text -> number
    if not pd.api.types.is_numeric_dtype(df["mileage"]):
        df["mileage"] = pd.to_numeric(
            df["mileage"].astype(str).str.replace(r"[^\d.]", "", regex=True), errors="coerce"
        )
    df["mileage"] = df["mileage"].clip(lower=0, upper=500_000)     # cap absurd values, keep the row

    # tidy text columns
    for c in ["brand", "model", "fuel_type", "engine", "transmission", "ext_col", "int_col"]:
        if c in df.columns:
            df[c] = df[c].astype("object").where(df[c].notna(), np.nan)
            df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)

    # inconsistent fuel labels
    df["fuel_type"] = df["fuel_type"].replace({"–": np.nan, "-": np.nan, "not supported": np.nan})
    df["fuel_type"] = df["fuel_type"].fillna("Unknown")

    # accident / clean title -> binary
    df["accident"] = df["accident"].map(
        lambda v: np.nan if pd.isna(v) else int("at least 1" in str(v).lower())
    )
    df["clean_title"] = df["clean_title"].map(lambda v: 1 if str(v).strip().lower() == "yes" else 0)

    return df

## 3. Feature Engineering


In [4]:
LUXURY_BRANDS = {
    "Porsche", "Ferrari", "Lamborghini", "Bentley", "Rolls-Royce", "Maserati", "McLaren", "Aston",
    "Bugatti", "Lotus", "Maybach", "Mercedes-Benz", "BMW", "Audi", "Land", "Jaguar", "Lexus",
    "Genesis", "Cadillac", "Tesla", "INFINITI", "Acura", "Lincoln", "Alfa", "Volvo",
}

def feature_engineering(df):
    """Creates new features from raw columns. No row dropping, no price usage."""
    df = df.copy()

    # --- engine text -> HP, litres, cylinders, flags
    eng = df["engine"].astype(str)
    df["hp"] = pd.to_numeric(eng.str.extract(r"(\d+\.?\d*)\s*HP", flags=re.I)[0], errors="coerce")
    df["liters"] = pd.to_numeric(eng.str.extract(r"(\d+\.?\d*)\s*L(?:iter)?\b", flags=re.I)[0], errors="coerce")
    cyl = pd.to_numeric(eng.str.extract(r"(\d+)\s*Cylinder", flags=re.I)[0], errors="coerce")
    cyl_v = pd.to_numeric(eng.str.extract(r"\b[VvIi](\d{1,2})\b")[0], errors="coerce")   # V6, V8, I4 ...
    df["cylinders"] = cyl.fillna(cyl_v)
    df["is_turbo"] = eng.str.contains("turbo|twin", case=False).astype(int)
    df["is_electric"] = eng.str.contains("electric|battery", case=False).astype(int)

    # --- age and usage
    df["car_age"] = (2025 - df["model_year"]).clip(lower=0)
    df["miles_per_year"] = df["mileage"] / df["car_age"].clip(lower=1)
    df["log_mileage"] = np.log1p(df["mileage"])

    # --- transmission simplification
    t = df["transmission"].astype(str).str.lower()
    df["trans_type"] = np.select(
        [t.str.contains("cvt"), t.str.contains("m/t|manual"), t.str.contains("a/t|auto|dual|dct|shift")],
        ["CVT", "Manual", "Automatic"], default="Other",
    )
    df["gears"] = pd.to_numeric(t.str.extract(r"(\d+)-speed")[0], errors="coerce")

    # --- brand grouping
    df["is_luxury"] = df["brand"].isin(LUXURY_BRANDS).astype(int)

    return df.drop(columns=["engine", "transmission"])

## 4. Transform, Normalize, and Scale Features



In [5]:
NUMERIC_FEATURES = [
    "model_year", "mileage", "log_mileage", "car_age", "miles_per_year",
    "hp", "liters", "cylinders", "gears",
    "is_turbo", "is_electric", "is_luxury", "accident", "clean_title",
]
CATEGORICAL_FEATURES = ["brand", "model", "fuel_type", "ext_col", "int_col", "trans_type"]
FINAL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES


def scale_features(df):
    """
    Final feature table. The actual encoding (OneHot / Ordinal) and scaling (StandardScaler) are done by the
    ColumnTransformer INSIDE the model Pipeline (see Section 6), so they are fitted on the train split only.
    This function guarantees the same columns / order / dtypes for train, validation, test and hidden data.
    """
    df = df.copy()
    for c in FINAL_FEATURES:
        if c not in df.columns:
            df[c] = np.nan
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce")
    df[CATEGORICAL_FEATURES] = df[CATEGORICAL_FEATURES].astype(str)
    return df[FINAL_FEATURES]

In [6]:

def preprocess_data(df):
    df = df.copy()
    df = data_cleaning(df)
    df = feature_engineering(df)
    df = scale_features(df)
    return df

## 5. Train / Validation / Test Split


In [7]:
def split_data(df):
    # Raw data is split FIRST (70 / 15 / 15) before any fitting, so nothing leaks between sets.
    # df still contains the target column; the returned dataframes do too.
    temp_df, test_split = train_test_split(df, test_size=0.15, random_state=RANDOM_STATE)
    train_split, validation_split = train_test_split(temp_df, test_size=0.1765, random_state=RANDOM_STATE)
    return train_split, validation_split, test_split


def remove_bad_training_rows(train_df):
    """Train split ONLY: drop exact duplicates and unrealistic prices. Never applied to val/test/hidden data."""
    train_df = train_df.drop_duplicates()
    train_df = train_df[(train_df[TARGET_COLUMN] >= 1_000) & (train_df[TARGET_COLUMN] <= 500_000)]
    return train_df


train_df, val_df, test_df = split_data(df)
print("Before cleaning train:", len(train_df), "| val:", len(val_df), "| test:", len(test_df))
train_df = remove_bad_training_rows(train_df)
print("After removing bad train rows:", len(train_df))

Before cleaning train: 2244 | val: 481 | test: 482
After removing bad train rows: 2237


## 6. Model Training, and Hyperparameter Tuning


In [8]:
# ---------- Build model matrices (cleaning + features on each split separately) ----------
X_train = preprocess_data(train_df.drop(columns=[TARGET_COLUMN])); y_train = train_df[TARGET_COLUMN]
X_val   = preprocess_data(val_df.drop(columns=[TARGET_COLUMN]));   y_val   = val_df[TARGET_COLUMN]
X_test  = preprocess_data(test_df.drop(columns=[TARGET_COLUMN]));  y_test  = test_df[TARGET_COLUMN]


def build_model(regressor, kind="tree"):
    """Pipeline = impute -> encode -> scale -> regressor. Target is trained on log(price) and converted
    back to dollars automatically (TransformedTargetRegressor), so predictions are always in US dollars."""
    if kind == "linear":
        num = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
        cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5))])
    else:
        num = Pipeline([("imp", SimpleImputer(strategy="median"))])
        cat = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                                               min_frequency=3, encoded_missing_value=-1))])
    pre = ColumnTransformer([("num", num, NUMERIC_FEATURES), ("cat", cat, CATEGORICAL_FEATURES)])
    return TransformedTargetRegressor(
        regressor=Pipeline([("prep", pre), ("reg", regressor)]),
        func=np.log1p, inverse_func=np.expm1,
    )


# ---------- Compare models on the validation set ----------
candidates = {
    "Ridge":        build_model(Ridge(alpha=3.0), kind="linear"),
    "RandomForest": build_model(RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)),
    "HistGB":       build_model(HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
}
for name, m in candidates.items():
    m.fit(X_train, y_train)
    print(f"{name:<13} val MAE = ${mean_absolute_error(y_val, m.predict(X_val)):,.0f}")

# ---------- Tune the best family (HistGB) on the VALIDATION set ----------
param_space = {
    "learning_rate":     [0.03, 0.05, 0.08, 0.1],
    "max_iter":          [200, 400, 600, 800],
    "max_depth":         [None, 6, 10],
    "max_leaf_nodes":    [15, 31, 63],
    "min_samples_leaf":  [5, 10, 20, 40],
    "l2_regularization": [0, 0.1, 1.0],
}
best_mae, best_params = np.inf, None
for params in ParameterSampler(param_space, n_iter=30, random_state=RANDOM_STATE):
    m = build_model(HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params))
    m.fit(X_train, y_train)
    mae = mean_absolute_error(y_val, m.predict(X_val))
    if mae < best_mae:
        best_mae, best_params, tuned_model = mae, params, m
print("\nBest params:", best_params)
print(f"Tuned HistGB val MAE = ${best_mae:,.0f}")


REFIT_ON_TRAIN_VAL = True
if REFIT_ON_TRAIN_VAL:
    model = build_model(HistGradientBoostingRegressor(random_state=RANDOM_STATE, **best_params))
    model.fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
else:
    model = tuned_model

Ridge         val MAE = $11,179
RandomForest  val MAE = $11,535
HistGB        val MAE = $10,204

Best params: {'min_samples_leaf': 10, 'max_leaf_nodes': 31, 'max_iter': 400, 'max_depth': 6, 'learning_rate': 0.1, 'l2_regularization': 1.0}
Tuned HistGB val MAE = $9,767


In [9]:
def predict_price(df):
    processed = preprocess_data(df)
    predictions = model.predict(processed)
    return predictions

## 7. Evaluate on the Held-Out Test Set


In [10]:
def evaluate_predictions(y_true, y_pred):
   # Standard regression metric report used throughout this notebook.
    # Both your own evaluation (Section 6) and the grading step (Section 8) use this function.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics = {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }
    print("--- Metrics ---")
    for name, value in metrics.items():
        print(f"{name:>10}: {value:.4f}")
    return metrics

In [11]:
print("TRAIN (tuned model, trained on train only)")
_ = evaluate_predictions(y_train, tuned_model.predict(X_train))
print("\nVALIDATION (tuned model, trained on train only)")
_ = evaluate_predictions(y_val, tuned_model.predict(X_val))

print("\nTEST (final model, via predict_price on RAW test rows)")
test_pred = predict_price(test_df.drop(columns=[TARGET_COLUMN]))
_ = evaluate_predictions(y_test, test_pred)

TRAIN (tuned model, trained on train only)
--- Metrics ---
      RMSE: 5389.6645
       MAE: 2595.0012
        R2: 0.9851

VALIDATION (tuned model, trained on train only)
--- Metrics ---
      RMSE: 28317.6101
       MAE: 9766.6478
        R2: 0.6524

TEST (final model, via predict_price on RAW test rows)
--- Metrics ---
      RMSE: 28757.1023
       MAE: 10190.0834
        R2: 0.7305


---
## 8. Standardized Evaluation

In [14]:
# =====================================================================
# ORGANIZER EVALUATION CELL
# =====================================================================
assert "predict_price" in dir(), (
    "predict_price(df) is not defined. Your submission must define this "
    "function in Section 6 (see instructions there)."
)

evaluation_df = pd.read_csv(EVALUATION_SPLIT_PATH)
X_hidden = evaluation_df.drop(columns=[TARGET_COLUMN])
y_hidden = evaluation_df[TARGET_COLUMN]

y_pred = predict_price(X_hidden)

evaluation_metrics = evaluate_predictions(y_hidden, y_pred)
pd.DataFrame([evaluation_metrics])

FileNotFoundError: [Errno 2] No such file or directory: './evaluation_split_task2.csv'